# Análise Experimental — STT + Extração Estruturada de Comandos Médicos

Este notebook documenta os resultados do protótipo de extração estruturada a partir de comandos de voz em português brasileiro.

O objetivo não é demonstrar um sistema de produção, mas analisar experimentalmente um pipeline mínimo:

```text
Áudio → Whisper STT → Transcrição → LLM → Schema Pydantic → Regras de domínio → JSON validado
```

Foram comparadas duas variantes:

| Variante | Descrição |
|---|---|
| A | LLM puro com validação estrutural via Pydantic |
| B | LLM + regras determinísticas de domínio médico |

A pergunta principal é: **a camada híbrida aumenta a robustez da extração estruturada?**

## 1. Carregamento dos dados

O notebook foi escrito para funcionar tanto quando está na raiz do projeto quanto dentro da pasta `notebooks/`.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

paths = {
    "stt": PROJECT_ROOT / "data" / "transcripts" / "stt_evaluation.json",
    "var_a": PROJECT_ROOT / "data" / "results" / "variante_a.json",
    "var_b": PROJECT_ROOT / "data" / "results" / "variante_b.json",
    "evaluation": PROJECT_ROOT / "data" / "results" / "evaluation.json",
}

for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

with open(paths["stt"], encoding="utf-8") as f:
    stt_data = json.load(f)

with open(paths["var_a"], encoding="utf-8") as f:
    var_a = json.load(f)

with open(paths["var_b"], encoding="utf-8") as f:
    var_b = json.load(f)

with open(paths["evaluation"], encoding="utf-8") as f:
    evaluation = json.load(f)

print("Dados carregados com sucesso.")
print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Casos STT: {stt_data['total_casos']}")
print(f"Casos Variante A: {var_a['total_casos']}")
print(f"Casos Variante B: {var_b['total_casos']}")

Dados carregados com sucesso.
Raiz do projeto: /home/CIN/mlfm/voice-command-extraction
Casos STT: 25
Casos Variante A: 25
Casos Variante B: 25


## 2. Avaliação do STT

O Whisper foi avaliado com duas leituras:

1. **WER/CER bruto:** comparação literal entre referência e transcrição.
2. **WER/CER normalizado:** comparação após normalizar números, unidades e aliases técnicos.

Essa distinção é importante porque `vinte` e `20` são diferentes para WER/CER, mas equivalentes para a extração semântica.

In [2]:
metricas = stt_data["metricas"]

df_stt_global = pd.DataFrame([
    {
        "Métrica": "WER",
        "Raw": metricas["wer_raw_medio"],
        "Normalizado": metricas["wer_normalizado_medio"],
        "Redução relativa (%)": round((1 - metricas["wer_normalizado_medio"] / metricas["wer_raw_medio"]) * 100, 1),
    },
    {
        "Métrica": "CER",
        "Raw": metricas["cer_raw_medio"],
        "Normalizado": metricas["cer_normalizado_medio"],
        "Redução relativa (%)": round((1 - metricas["cer_normalizado_medio"] / metricas["cer_raw_medio"]) * 100, 1),
    },
])

df_stt_global

,Métrica,Raw,Normalizado,Redução relativa (%)
0,WER,0.5967,0.3273,45.1
1,CER,0.3411,0.1337,60.8


### Interpretação

A normalização reduziu substancialmente o erro aparente do STT. Isso indica que parte relevante do WER/CER bruto estava ligada à representação textual — por exemplo, número por extenso versus algarismo — e não necessariamente à perda de significado.

Essa análise é importante porque o objetivo final do componente não é reproduzir a transcrição literalmente, mas extrair campos estruturados corretamente.

In [3]:
df_stt = pd.DataFrame(stt_data["resultados"])

df_stt_by_source = (
    df_stt
    .groupby("fonte")[["wer_raw", "wer_normalizado", "cer_raw", "cer_normalizado"]]
    .mean()
    .round(4)
)

df_stt_by_category = (
    df_stt
    .groupby("categoria")[["wer_raw", "wer_normalizado", "cer_raw", "cer_normalizado"]]
    .mean()
    .round(4)
)

df_stt_by_source

,wer_raw,wer_normalizado,cer_raw,cer_normalizado
fonte,,,,
gtts,0.5444,0.1789,0.3378,0.0542
voz_humana,0.6750,0.5500,0.3461,0.2530


In [4]:
df_stt_by_category

,wer_raw,wer_normalizado,cer_raw,cer_normalizado
categoria,,,,
ambiguo,0.6958,0.5000,0.2927,0.3051
erro_stt,0.6500,0.4833,0.4119,0.1582
fora_de_faixa,0.5083,0.2500,0.4335,0.0744
incompleto,0.4444,0.0000,0.0536,0.0000
unidade_omitida,0.5458,0.1875,0.3561,0.0509
valido_simples,0.6639,0.4167,0.3984,0.1648


### Casos com maior erro de STT

Os maiores erros ajudam a diferenciar falhas recuperáveis de falhas irrecuperáveis. Em particular, quando o STT altera o valor numérico, o restante do pipeline não deve tentar adivinhar a intenção original.

In [5]:
df_stt_cases = df_stt[[
    "id", "categoria", "fonte",
    "referencia", "transcricao_whisper",
    "wer_raw", "wer_normalizado"
]].copy()

df_stt_cases = df_stt_cases.sort_values("wer_normalizado", ascending=False)
pd.set_option("display.max_colwidth", 100)

df_stt_cases.head(8)

,id,categoria,fonte,referencia,transcricao_whisper,wer_raw,wer_normalizado
13,14,ambiguo,voz_humana,ajustar para vinte,Azueta paraə vwan Ontiniz,1.3333,1.3333
2,3,valido_simples,voz_humana,reduzir pressão inspiratória para quinze centímetros de água,Reduzir a impressão inspiradora para 15 cm de água.,0.7500,1.0000
12,13,ambiguo,voz_humana,fio dois tá baixo,Fio 2 tabaixo,0.7500,0.6667
22,23,erro_stt,gtts,frequência respiratória vinte i rpm,Frequência respiratória 20RPM.,0.6000,0.6000
0,1,valido_simples,voz_humana,aumentar frequência respiratória para vinte rpm,Aumentar frequência respirantória para vinte e repem.,0.5000,0.5000
18,19,fora_de_faixa,gtts,fio dois cem por cento,Fio 200%,0.8000,0.5000
1,2,valido_simples,voz_humana,definir volume corrente em quinhentos mililitros,Definível um ecorrente em 500 ml.,0.8333,0.5000
3,4,valido_simples,voz_humana,aumentar fio dois para sessenta por cento,"Almentar o fio 2,60%.",1.0000,0.5000


## 3. Saídas das variantes

A Variante A representa uma baseline baseada no LLM. A Variante B usa a mesma extração inicial, mas aplica regras de domínio para normalizar unidade, detectar inferência, validar faixa clínica e ajustar `confidence`/`requires_confirmation`.

A diferença central não é apenas “acertar mais campos”, mas tornar decisões sensíveis mais auditáveis.

In [6]:
def variant_to_frame(data, variant):
    rows = []
    for item in data["resultados"]:
        if variant == "A":
            saida = item.get("saida") or {}
        else:
            saida = item.get("saida_final") or {}

        rows.append({
            "id": item["id"],
            "categoria": item["categoria"],
            "fonte": item["fonte"],
            "transcript": item["transcript_whisper"],
            "intent": saida.get("intent"),
            "parameter": saida.get("parameter"),
            "value": saida.get("value"),
            "unit": saida.get("unit"),
            "status": saida.get("status"),
            "confidence": saida.get("confidence"),
            "requires_confirmation": saida.get("requires_confirmation"),
            "n_validation_errors": len(saida.get("validation_errors", [])) if saida else None,
            "erro_pipeline": item.get("erro"),
        })
    return pd.DataFrame(rows)

df_a = variant_to_frame(var_a, "A")
df_b = variant_to_frame(var_b, "B")

display(df_a.head())
display(df_b.head())

,id,categoria,fonte,transcript,intent,parameter,value,unit,status,confidence,requires_confirmation,n_validation_errors,erro_pipeline
0,1,valido_simples,voz_humana,Aumentar frequência respirantória para vinte e repem.,aumentar_parametro,frequencia_respiratoria,20.0,rpm,ok,high,False,0,None
1,2,valido_simples,voz_humana,Definível um ecorrente em 500 ml.,definir_parametro,volume_corrente,500.0,mL,ok,high,False,0,None
2,3,valido_simples,voz_humana,Reduzir a impressão inspiradora para 15 cm de água.,reduzir_parametro,pressao_inspiratoria,15.0,cmH2O,ok,high,False,0,None
3,4,valido_simples,voz_humana,"Almentar o fio 2,60%.",aumentar_parametro,fio2,60.0,%,ok,high,False,0,None
4,5,valido_simples,gtts,diminuir frequência para 12 RPM.,reduzir_parametro,frequencia_respiratoria,12.0,rpm,ok,high,False,0,None


,id,categoria,fonte,transcript,intent,parameter,value,unit,status,confidence,requires_confirmation,n_validation_errors,erro_pipeline
0,1,valido_simples,voz_humana,Aumentar frequência respirantória para vinte e repem.,aumentar_parametro,frequencia_respiratoria,20.0,rpm,ok,low,True,0,None
1,2,valido_simples,voz_humana,Definível um ecorrente em 500 ml.,definir_parametro,volume_corrente,500.0,mL,ok,high,False,0,None
2,3,valido_simples,voz_humana,Reduzir a impressão inspiradora para 15 cm de água.,reduzir_parametro,pressao_inspiratoria,15.0,cmH2O,ok,high,False,0,None
3,4,valido_simples,voz_humana,"Almentar o fio 2,60%.",aumentar_parametro,fio2,60.0,%,ok,high,False,0,None
4,5,valido_simples,gtts,diminuir frequência para 12 RPM.,reduzir_parametro,frequencia_respiratoria,12.0,rpm,ok,high,False,0,None


In [7]:
status_comparison = pd.DataFrame({
    "Variante A": df_a["status"].value_counts(),
    "Variante B": df_b["status"].value_counts(),
}).fillna(0).astype(int)

confidence_comparison = pd.DataFrame({
    "Variante A": df_a["confidence"].value_counts(),
    "Variante B": df_b["confidence"].value_counts(),
}).fillna(0).astype(int)

status_comparison

,Variante A,Variante B
status,,
ok,14,14
incompleto,7,7
fora_de_faixa,4,4


In [8]:
confidence_comparison

,Variante A,Variante B
confidence,,
high,20,11
low,5,14


### Observação sobre `fora_de_faixa`

A detecção de valores fora de faixa não mudou numericamente entre as variantes. Isso é uma observação importante, não uma falha.

O LLM foi capaz de detectar casos extremos. A diferença é que, na Variante A, essa decisão depende do modelo; na Variante B, ela é verificada por regra determinística. Portanto, o ganho da Variante B está mais em garantia e previsibilidade do que em mudança numérica nessa categoria específica.

## 4. Avaliação comparativa

A avaliação compara as saídas das variantes com o gabarito estruturado. As métricas incluem taxa de schema válido, erro crítico, acurácia por campo e F1 macro por intent.

In [9]:
a = evaluation["variante_a"]
b = evaluation["variante_b"]

df_summary = pd.DataFrame([
    {
        "Métrica": "Taxa schema válido",
        "Variante A": a["taxa_schema_valido"],
        "Variante B": b["taxa_schema_valido"],
        "Delta": b["taxa_schema_valido"] - a["taxa_schema_valido"],
    },
    {
        "Métrica": "Taxa erro crítico",
        "Variante A": a["taxa_erro_critico"],
        "Variante B": b["taxa_erro_critico"],
        "Delta": b["taxa_erro_critico"] - a["taxa_erro_critico"],
    },
    {
        "Métrica": "Acurácia média",
        "Variante A": a["acuracia_media"],
        "Variante B": b["acuracia_media"],
        "Delta": b["acuracia_media"] - a["acuracia_media"],
    },
    {
        "Métrica": "F1 macro intent",
        "Variante A": a["f1_macro_intent"],
        "Variante B": b["f1_macro_intent"],
        "Delta": b["f1_macro_intent"] - a["f1_macro_intent"],
    },
])

df_summary

,Métrica,Variante A,Variante B,Delta
0,Taxa schema válido,1.0000,1.0000,0.0000
1,Taxa erro crítico,0.0400,0.0400,0.0000
2,Acurácia média,0.8343,0.8800,0.0457
3,F1 macro intent,0.5307,0.5307,0.0000


In [10]:
fields = sorted(a["acuracia_por_campo"].keys())

df_fields = pd.DataFrame([
    {
        "Campo": field,
        "Variante A": a["acuracia_por_campo"][field],
        "Variante B": b["acuracia_por_campo"][field],
        "Delta": b["acuracia_por_campo"][field] - a["acuracia_por_campo"][field],
    }
    for field in fields
])

df_fields.sort_values("Delta", ascending=False)

,Campo,Variante A,Variante B,Delta
0,confidence,0.60,0.88,0.28
3,requires_confirmation,0.84,0.88,0.04
1,intent,0.84,0.84,0.00
2,parameter,1.00,1.00,0.00
4,status,0.80,0.80,0.00
5,unit,0.84,0.84,0.00
6,value,0.92,0.92,0.00


### Interpretação dos resultados

O principal ganho da Variante B aparece em `confidence`, que aumenta de 60% para 88%. Esse resultado é coerente com a estratégia híbrida: a camada de regras melhora decisões que não deveriam depender apenas da autodeclaração do LLM.

O ganho em `requires_confirmation` também é relevante, ainda que menor. Ele mostra que a Variante B detecta situações em que a unidade foi inferida ou em que há maior risco operacional.

Campos como `intent`, `parameter`, `value`, `unit` e `status` não mudaram. Isso sugere que o LLM já foi forte na extração semântica principal, enquanto as regras contribuíram principalmente para consistência, segurança e confiabilidade.

## 5. F1 por intent

O F1 macro ficou baixo porque algumas classes aparecem pouco no dataset. Em datasets pequenos, classes raras têm impacto desproporcional no macro F1.

In [11]:
df_f1 = pd.DataFrame([
    {
        "intent": intent,
        "precision_A": a["f1_por_intent"][intent]["precision"],
        "recall_A": a["f1_por_intent"][intent]["recall"],
        "f1_A": a["f1_por_intent"][intent]["f1"],
        "precision_B": b["f1_por_intent"][intent]["precision"],
        "recall_B": b["f1_por_intent"][intent]["recall"],
        "f1_B": b["f1_por_intent"][intent]["f1"],
    }
    for intent in a["f1_por_intent"]
])

df_f1

,intent,precision_A,recall_A,f1_A,precision_B,recall_B,f1_B
0,aumentar_parametro,1.00,0.7143,0.8333,1.00,0.7143,0.8333
1,reduzir_parametro,0.75,1.0000,0.8571,0.75,1.0000,0.8571
2,definir_parametro,1.00,0.9286,0.9630,1.00,0.9286,0.9630
3,consultar_parametro,0.00,0.0000,0.0000,0.00,0.0000,0.0000
4,desconhecido,0.00,0.0000,0.0000,0.00,0.0000,0.0000


As classes ```aumentar_parametro```, ```reduzir_parametro``` e ```definir_parametro``` têm F1 acima de 0.83, o que indica boa performance nas intenções mais comuns do domínio. As classes ```consultar_parametro``` e ```desconhecido``` têm F1 zero porque aparecem raramente no dataset — um caso cada.

Com 25 exemplos, o macro F1 é penalizado de forma desproporcional por classes sub-representadas. Essa limitação é esperada e documentada, não um erro do modelo.

## 6. Análise de erros críticos

Erro crítico foi definido como uma situação em que o sistema aceita como `ok` um caso que deveria ser bloqueado, ou erra valor/parâmetro em caso válido.

O erro persistente mais relevante está associado à etapa de STT: quando a transcrição altera semanticamente o valor, a recuperação posterior se torna insegura.

In [12]:
details_a = pd.DataFrame(a["detalhes"])
details_b = pd.DataFrame(b["detalhes"])

critical_a = details_a[details_a["erro_critico"] == True]
critical_b = details_b[details_b["erro_critico"] == True]

print("Erros críticos A:", critical_a["id"].tolist())
print("Erros críticos B:", critical_b["id"].tolist())

df_stt[df_stt["id"].isin(sorted(set(critical_a["id"].tolist() + critical_b["id"].tolist())))]

Erros críticos A: [19]
Erros críticos B: [19]


,id,categoria,fonte,audio,referencia,transcricao_whisper,wer,cer,referencia_normalizada,transcricao_normalizada,wer_raw,cer_raw,wer_normalizado,cer_normalizado
18,19,fora_de_faixa,gtts,data/audio/cmd_19.mp3,fio dois cem por cento,Fio 200%,0.8,0.8182,fio2 100 por 100,fio200 por 100,0.8,0.8182,0.5,0.125


O único erro crítico identificado (caso 19) teve origem no STT: "fio dois cem por cento" foi transcrito como "Fio 200%", alterando o valor numérico na fonte.

O LLM, recebendo "200%", corretamente sinalizou ```fora_de_faixa``` — mas o valor real era 100%, que estaria dentro da faixa. **Nenhuma regra de domínio consegue recuperar um valor corrompido na transcrição**. Esse caso ilustra que o limite fundamental do pipeline é a qualidade do STT.

## 7. Conclusão

### 7.1 Sobre o STT

O **Whisper** base processou todos os 25 áudios sem falha técnica, mas apresentou erros frequentes em termos médicos, siglas e fala natural rápida. A análise de WER/CER em duas versões — bruta e normalizada — revelou que parte relevante do erro aparente era artefato de representação: o modelo entendia o conteúdo, mas o escrevia de forma diferente da referência ("quinhentos" → "500", "rpm" → "RPM"). Após normalização, o WER caiu 45% e o CER caiu 61%.

Isso tem uma implicação direta: **avaliar STT em domínio especializado sem normalização superestima os erros reais e pode levar a conclusões equivocadas sobre a qualidade do modelo.** A etapa de normalização é necessária para medir o que de fato importa para a extração semântica.

O limite do STT ficou evidente no caso 19, onde "fio dois cem por cento" foi transcrito como "Fio 200%". O valor numérico foi corrompido na fonte. Esse tipo de erro é irrecuperável por qualquer camada downstream — e estabelece um teto para a qualidade de tudo que vem depois.

### 7.2 Sobre a extração estruturada

O LLM demonstrou robustez notável a transcrições corrompidas. Casos como "Definível um ecorrente em 500 ml" e "Almentar o fio 2,60%" foram extraídos corretamente. A acurácia em `parameter` foi de 100% em ambas as variantes, o que sugere que o modelo consegue inferir o parâmetro médico mesmo quando a transcrição está parcialmente comprometida.

Onde o LLM mostrou limitação foi em campos que dependem de conhecimento externo ao texto: ele não sabe quando inferiu uma unidade versus quando ela foi dita explicitamente, e não tem como verificar faixas clínicas sem que esse conhecimento esteja embutido no prompt. Essas lacunas são estruturais e não podem ser resolvidas por um prompt melhor, mas por uma camada diferente de validação.

### 7.3 Sobre a comparação entre variantes

A Variante B não melhorou a extração semântica bruta. `intent`, `parameter`, `value`, `unit` e `status` ficaram iguais entre as duas variantes. O LLM já era forte nesses campos.

O que a Variante B melhorou foram decisões que dependem de política e consistência de domínio: `confidence` subiu 28 pontos percentuais (de 60% para 88%) e `requires_confirmation` subiu 4 pontos percentuais. Esses campos representam o quanto o sistema confia no que extraiu — e essa distinção é clinicamente relevante.

Isso sugere que LLM e regras determinísticas não competem: eles cobrem domínios de competência diferentes. O LLM interpreta linguagem natural com flexibilidade. As regras verificam, auditam e garantem limites que o modelo não tem como conhecer por si só.

### 7.4 Implicação para sistemas reais

Em sistemas médicos, uma inferência silenciosa de unidade pode resultar em um parâmetro aplicado sem confirmação do operador. Um valor fora de faixa aceito como válido pode colocar um paciente em risco. A arquitetura híbrida não elimina esses riscos — erros semânticos do STT continuam sendo irrecuperáveis — porém, torna o sistema mais previsível, auditável e seguro.

A principal conclusão do experimento é que a abordagem mais robusta não é "LLM puro", mas um pipeline em que o LLM interpreta a linguagem e regras determinísticas garantem os limites de segurança. Em domínios críticos, a previsibilidade e a auditabilidade da decisão importam tanto quanto a acurácia da extração.